# STRING — Multi-Species Protein–Protein Interaction KG Processing

**Species processed:**

| Tax ID | Species | STRING file |
|--------|---------|-------------|
| 9606 | *Homo sapiens* | `9606.protein.links.detailed.v12.0.txt` |
| 10090 | *Mus musculus* | `10090.protein.links.detailed.v12.0.txt` |
| 7227 | *Drosophila melanogaster* | `7227.protein.links.detailed.v12.0.txt` |
| 6239 | *Caenorhabditis elegans* | `6239.protein.links.detailed.v12.0.txt` |
| 7955 | *Danio rerio* | `7955.protein.links.detailed.v12.0.txt` |
| 4932 | *Saccharomyces cerevisiae* | `4932.protein.links.detailed.v12.0.txt` |

**Filter:** `experimental > 0` (direct biochemical / genetic evidence only).  
**Protein ID chain:** `<taxid>.ENSP... → strip prefix → gene symbol via NCBI gene_info → description`  
For *H. sapiens* only: additional UniProt AC annotation via Ensembl uniprot TSV.

All outputs written to `OUT_PATH`. One CSV per species.



## 0. Path Configuration

In [1]:
import os

# ── Edit only these four lines ───────────────────────────────────────────────
BASE_PATH    = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
DB_BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/"
STRING_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/string/"   # STRING v12.0 raw .txt files (one per species)
OUT_PATH     = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/"      # all output KG CSVs land here
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUT_PATH, exist_ok=True)
print("STRING_PATH  :", STRING_PATH)
print("DB_BASE_PATH :", DB_BASE_PATH)
print("OUT_PATH     :", OUT_PATH)


STRING_PATH  : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/string/
DB_BASE_PATH : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/databases_for_mapping/
OUT_PATH     : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/string/


## 1. Imports

In [2]:
import os
import numpy as np
import pandas as pd
from glob import glob


## 2. Shared Helpers

In [3]:
KG_DESIRED_COLS = [
    "Head", "Relation", "Tail",
    "Head_type", "Tail_type",
    "KG_Source",
    "Head_detail_name",
    "Tail_detail_name",
    "Head_ID_IS", "Tail_ID_IS",
    "Species",
]

def reorder(df):
    return df[[c for c in KG_DESIRED_COLS if c in df.columns]]

def load_string(tax_id):
    """
    Load STRING detailed links file for a given tax_id.
    """
    fname = f"{tax_id}.protein.links.detailed.v12.0.txt"
    fpath = os.path.join(STRING_PATH, fname)
    df = pd.read_csv(fpath, sep=r"\s+", engine="python")

    prefix = f"{tax_id}."
    df["protein1"] = df["protein1"].str.replace(f"^{tax_id}\\.", "", regex=True)
    df["protein2"] = df["protein2"].str.replace(f"^{tax_id}\\.", "", regex=True)

    df = df[df["experimental"] > 0].copy()
    return df


## 3. Reference Dictionaries

### 3.1 UniProt: Ensembl ENSP → UniProt AC (Human only)

In [4]:
# Human ENSP → UniProt AC (Swiss-Prot preferred over TrEMBL)
Uniprot_file = pd.read_csv(
    os.path.join(DB_BASE_PATH, "ENSEMBL", "Homo_sapiens.GRCh38.113.uniprot.tsv"),
    sep="\t"
)
sp   = Uniprot_file[Uniprot_file["db_name"] == "Uniprot/SWISSPROT"]
trembl = Uniprot_file[Uniprot_file["db_name"] == "Uniprot/SPTREMBL"]

# TrEMBL first, overwrite with SwissProt → reviewed entries always win
Uniprot_file_dict = {
    **dict(zip(trembl["protein_stable_id"], trembl["xref"])),
    **dict(zip(sp["protein_stable_id"],     sp["xref"])),
}
del Uniprot_file, sp, trembl
print(f"Uniprot_file_dict (ENSP → AC): {len(Uniprot_file_dict):,}")


Uniprot_file_dict (ENSP → AC): 120,162


### 3.2 UniProt AC → RecName

In [6]:
Uniprot_Recname = pd.read_csv(
    os.path.join(DB_BASE_PATH, "uniprot", "Uniprot_ID_Recname.csv")
)
Uniprot_Recname_dict = dict(zip(Uniprot_Recname["AC"], Uniprot_Recname["RecName"]))
del Uniprot_Recname
print(f"Uniprot_Recname_dict (AC → RecName): {len(Uniprot_Recname_dict):,}")


Uniprot_Recname_dict (AC → RecName): 794,151


### 3.3 NCBI Gene Info — per species (Ensembl/protein ID → Symbol → Description)

In [ ]:
# Each NCBI gene_info file: Ensembl protein stable ID extracted from dbXrefs
# Maps: ENSP/protein_ID → gene Symbol → description

def load_ncbi_gene_dicts(gene_info_path):
    """
    Returns (symbol_desc_dict, ensp_symbol_dict) for a given NCBI gene_info file.
    symbol_desc_dict : gene symbol → description
    ensp_symbol_dict : Ensembl protein ID → gene symbol (extracted from dbXrefs)
    """
    gi = pd.read_csv(gene_info_path, sep="\t", low_memory=False)

    symbol_desc_dict = dict(zip(gi["Symbol"], gi["description"]))

    # Extract Ensembl protein IDs from dbXrefs (format: Ensembl:ENSPxxx|...)
    ensp_rows = []
    for _, row in gi.iterrows():
        refs = str(row["dbXrefs"])
        for token in refs.split("|"):
            if token.startswith("Ensembl:ENSP") or token.startswith("Ensembl:WBP") or \
               token.startswith("Ensembl:FB") or token.startswith("Ensembl:"):
                ensp_rows.append((token.replace("Ensembl:", ""), row["Symbol"]))
    ensp_symbol_dict = dict(ensp_rows)

    return symbol_desc_dict, ensp_symbol_dict

# Human
human_symbol_desc, human_ensp_sym = load_ncbi_gene_dicts(
    os.path.join(DB_BASE_PATH, "NCBI", "Homo_sapiens.gene_info")
)
print(f"Human  — symbol_desc: {len(human_symbol_desc):,} | ensp_sym: {len(human_ensp_sym):,}")

# Mouse — also has gProfiler mapping
mouse_symbol_desc, mouse_ensp_sym = load_ncbi_gene_dicts(
    os.path.join(DB_BASE_PATH, "NCBI", "Mus_musculus.gene_info")
)
print(f"Mouse  — symbol_desc: {len(mouse_symbol_desc):,} | ensp_sym: {len(mouse_ensp_sym):,}")

# Drosophila
droso_symbol_desc, droso_ensp_sym = load_ncbi_gene_dicts(
    os.path.join(DB_BASE_PATH, "NCBI", "Drosophila_melanogaster.gene_info")
)
print(f"Droso  — symbol_desc: {len(droso_symbol_desc):,} | ensp_sym: {len(droso_ensp_sym):,}")

# C. elegans
cele_symbol_desc, cele_ensp_sym = load_ncbi_gene_dicts(
    os.path.join(DB_BASE_PATH, "NCBI", "Caenorhabditis_elegans.gene_info")
)
print(f"Cele   — symbol_desc: {len(cele_symbol_desc):,} | ensp_sym: {len(cele_ensp_sym):,}")

# Zebrafish
zebra_symbol_desc, zebra_ensp_sym = load_ncbi_gene_dicts(
    os.path.join(DB_BASE_PATH, "NCBI", "Danio_rerio.gene_info")
)
print(f"Zebra  — symbol_desc: {len(zebra_symbol_desc):,} | ensp_sym: {len(zebra_ensp_sym):,}")

# Yeast
yeast_symbol_desc, yeast_ensp_sym = load_ncbi_gene_dicts(
    os.path.join(DB_BASE_PATH, "NCBI", "Saccharomyces_cerevisiae.gene_info")
)
print(f"Yeast  — symbol_desc: {len(yeast_symbol_desc):,} | ensp_sym: {len(yeast_ensp_sym):,}")


## 4. Core Processing Function
One function handles all species. For Human, an additional UniProt AC annotation
layer is applied (ENSP → UniProt AC → RecName).


In [ ]:
def process_string_species(
    tax_id,
    species_label,
    ensp_sym_dict,
    symbol_desc_dict,
    out_filename,
    extra_ensp_sym_dict=None,   # e.g. gProfiler for mouse
):
    """
    Load, filter, map, and save one STRING species file.

    Parameters
    ----------
    tax_id            : str  e.g. '9606'
    species_label     : str  e.g. 'H.sapiens'
    ensp_sym_dict     : dict  Ensembl protein ID → gene symbol (from NCBI gene_info)
    symbol_desc_dict  : dict  gene symbol → description
    out_filename      : str  output CSV filename (no path)
    extra_ensp_sym_dict : dict or None  additional ENSP→symbol map (merged with priority)
    """
    print(f"\n{'='*60}")
    print(f"Processing: {species_label} (tax_id={tax_id})")

    df = load_string(tax_id)
    print(f"  After experimental > 0 filter: {len(df):,} rows")

    df = df.rename(columns={"protein1": "Head", "protein2": "Tail"})

    # Merge ENSP→symbol maps (extra has higher coverage, e.g. gProfiler)
    combined_ensp_sym = dict(ensp_sym_dict)
    if extra_ensp_sym_dict:
        combined_ensp_sym.update(extra_ensp_sym_dict)   # extra overwrites on conflict

    # Map ENSP protein IDs → gene symbols
    df["Head_sym"] = df["Head"].map(combined_ensp_sym)
    df["Tail_sym"] = df["Tail"].map(combined_ensp_sym)
    df = df[df["Head_sym"].notna() & df["Tail_sym"].notna()]
    print(f"  After symbol mapping: {len(df):,} rows")

    # Annotate with gene descriptions
    df["Head_detail_name"] = df["Head_sym"].map(symbol_desc_dict)
    df["Tail_detail_name"] = df["Tail_sym"].map(symbol_desc_dict)
    df = df[df["Head_detail_name"].notna() & df["Tail_detail_name"].notna()]
    print(f"  After description annotation: {len(df):,} rows")

    # Swap: gene symbol → Head/Tail; original ENSP stays recorded in Head_sym/Tail_sym
    df["Head_Alt_ID"] = df["Head"]   # original ENSP
    df["Tail_Alt_ID"] = df["Tail"]
    df["Head"] = df["Head_sym"]
    df["Tail"] = df["Tail_sym"]
    df.drop(columns=["Head_sym", "Tail_sym"], inplace=True)

    # KG metadata
    df["Head_type"]  = "Protein"
    df["Tail_type"]  = "Protein"
    df["Relation"]   = "Protein_Protein"
    df["Head_ID_IS"] = "NCBI_Symbol"
    df["Tail_ID_IS"] = "NCBI_Symbol"
    df["KG_Source"]  = "STRING"
    df["Species"]    = species_label

    df = reorder(df)

    out_path = os.path.join(OUT_PATH, out_filename)
    df.to_csv(out_path, index=False)
    print(f"  Saved: {out_filename}  ({len(df):,} triples)")
    return df


### 4.1 Human: additional UniProt annotation layer
For *H. sapiens*, we can also annotate with UniProt AC + RecName (higher-quality protein names)
since we have the Ensembl→UniProt TSV. This produces an alternative richer output.


In [ ]:
def process_string_human_uniprot():
    """
    Human STRING with full UniProt AC → RecName annotation chain.
    Head/Tail = UniProt AC; Head/Tail_detail_name = RecName.
    Saved separately as Protein_Protein_STRING_Human_UniProt.csv.
    """
    print("\nProcessing: H.sapiens (UniProt AC annotation)")
    df = load_string("9606")
    print(f"  After experimental > 0 filter: {len(df):,} rows")

    df = df.rename(columns={"protein1": "Head", "protein2": "Tail"})

    # Map ENSP → UniProt AC
    df["Head_AC"] = df["Head"].map(Uniprot_file_dict)
    df["Tail_AC"] = df["Tail"].map(Uniprot_file_dict)
    df = df[df["Head_AC"].notna() & df["Tail_AC"].notna()]
    print(f"  After UniProt AC mapping: {len(df):,} rows")

    # Map UniProt AC → RecName
    df["Head_detail_name"] = df["Head_AC"].map(Uniprot_Recname_dict)
    df["Tail_detail_name"] = df["Tail_AC"].map(Uniprot_Recname_dict)
    df = df[df["Head_detail_name"].notna() & df["Tail_detail_name"].notna()]
    print(f"  After RecName annotation: {len(df):,} rows")

    # Swap: UniProt AC → Head/Tail; original ENSP saved
    df["Head_Alt_ID"] = df["Head"]
    df["Tail_Alt_ID"] = df["Tail"]
    df["Head"] = df["Head_AC"]
    df["Tail"] = df["Tail_AC"]
    df.drop(columns=["Head_AC", "Tail_AC"], inplace=True)

    df["Head_type"]  = "Protein"
    df["Tail_type"]  = "Protein"
    df["Relation"]   = "Protein_Protein"
    df["Head_ID_IS"] = "Uniprot_ID"
    df["Tail_ID_IS"] = "Uniprot_ID"
    df["KG_Source"]  = "STRING"
    df["Species"]    = "H.sapiens"

    df = reorder(df)
    out = os.path.join(OUT_PATH, "Protein_Protein_STRING_Human_UniProt.csv")
    df.to_csv(out, index=False)
    print(f"  Saved: Protein_Protein_STRING_Human_UniProt.csv  ({len(df):,} triples)")
    return df


## 5. Run All Species

### 5.1 *Homo sapiens* (gene symbol + UniProt)

In [ ]:
human_df = process_string_species(
    tax_id          = "9606",
    species_label   = "H.sapiens",
    ensp_sym_dict   = human_ensp_sym,
    symbol_desc_dict= human_symbol_desc,
    out_filename    = "Protein_Protein_STRING_Human.csv",
)
human_df.head(3)


In [ ]:
# Also save the richer UniProt-annotated version for human
human_uniprot_df = process_string_human_uniprot()
human_uniprot_df.head(3)


### 5.2 *Mus musculus* (gProfiler + NCBI)

In [ ]:
mouse_df = process_string_species(
    tax_id               = "10090",
    species_label        = "M.musculus",
    ensp_sym_dict        = mouse_ensp_sym,
    symbol_desc_dict     = mouse_symbol_desc,
    out_filename         = "Protein_Protein_STRING_Mouse.csv",
    extra_ensp_sym_dict  = gpro_mouse_dict,   # gProfiler has higher coverage
)
mouse_df.head(3)


### 5.3 *Drosophila melanogaster*

In [ ]:
droso_df = process_string_species(
    tax_id          = "7227",
    species_label   = "D.melanogaster",
    ensp_sym_dict   = droso_ensp_sym,
    symbol_desc_dict= droso_symbol_desc,
    out_filename    = "Protein_Protein_STRING_Droso.csv",
)
droso_df.head(3)


### 5.4 *Caenorhabditis elegans*

In [ ]:
cele_df = process_string_species(
    tax_id          = "6239",
    species_label   = "C.elegans",
    ensp_sym_dict   = cele_ensp_sym,
    symbol_desc_dict= cele_symbol_desc,
    out_filename    = "Protein_Protein_STRING_Cele.csv",
)
cele_df.head(3)


### 5.5 *Danio rerio* (Zebrafish)

In [ ]:
zebra_df = process_string_species(
    tax_id          = "7955",
    species_label   = "D.rerio",
    ensp_sym_dict   = zebra_ensp_sym,
    symbol_desc_dict= zebra_symbol_desc,
    out_filename    = "Protein_Protein_STRING_Zebra.csv",
)
zebra_df.head(3)


### 5.6 *Saccharomyces cerevisiae* (Yeast)

In [ ]:
yeast_df = process_string_species(
    tax_id          = "4932",
    species_label   = "S.cerevisiae",
    ensp_sym_dict   = yeast_ensp_sym,
    symbol_desc_dict= yeast_symbol_desc,
    out_filename    = "Protein_Protein_STRING_Yeast.csv",
)
yeast_df.head(3)


## 6. Combined Multi-Species File

In [ ]:
all_species = [human_df, mouse_df, droso_df, cele_df, zebra_df, yeast_df]
combined = pd.concat(all_species, ignore_index=True)

combined.to_csv(os.path.join(OUT_PATH, "Protein_Protein_STRING_ALL_Species.csv"), index=False)
print(f"Combined rows: {len(combined):,}")
print(combined["Species"].value_counts().to_string())
del all_species, combined


## 7. Summary Audit

In [ ]:
cols_audit = ["Relation", "Head_type", "Tail_type", "KG_Source", "Head_ID_IS", "Tail_ID_IS", "Species"]
rows = []
for fp in sorted(glob(os.path.join(OUT_PATH, "*.csv"))):
    try:
        tmp = pd.read_csv(fp)
        row = {"File": os.path.basename(fp), "Triples": len(tmp)}
        for col in cols_audit:
            if col in tmp.columns:
                row[col] = set(tmp[col].dropna().unique())
        rows.append(row)
    except Exception as e:
        print(f"  Error: {fp} — {e}")

audit_df = pd.concat([pd.DataFrame([r]) for r in rows], ignore_index=True)
display(audit_df)
print(f"\nTotal triples across all files: {audit_df['Triples'].sum():,}")
